# VN History RAG Corpus Builder — CLEAN Colab Version

Bản này làm lại từ đầu để tránh lỗi `not defined`.

Chạy theo thứ tự từ trên xuống dưới, tốt nhất dùng **Runtime → Run all**.

Output chính:

```text
/content/drive/MyDrive/vn_history_model_backups/rag_corpus_vn_history/processed/vn_history_rag_chunks.jsonl
```

Notebook này chỉ build corpus JSONL/CSV. Chưa build FAISS.

## 0. Mount Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 1. Cài thư viện

In [2]:
!pip install -q datasets pandas tqdm beautifulsoup4 requests pyarrow

## 2. Imports và cấu hình

In [3]:
from pathlib import Path
import json
import re
import hashlib
import time
from typing import Dict, List, Any, Optional
from tqdm.auto import tqdm
import pandas as pd
from IPython.display import display

DRIVE_ROOT = Path("/content/drive/MyDrive/vn_history_model_backups")

PROJECT_DIR = DRIVE_ROOT / "rag_corpus_vn_history"
RAW_DIR = PROJECT_DIR / "raw"
PROCESSED_DIR = PROJECT_DIR / "processed"
LOG_DIR = PROJECT_DIR / "logs"

for p in [PROJECT_DIR, RAW_DIR, PROCESSED_DIR, LOG_DIR]:
    p.mkdir(parents=True, exist_ok=True)

OUTPUT_JSONL = PROCESSED_DIR / "vn_history_rag_chunks.jsonl"
OUTPUT_CSV = PROCESSED_DIR / "vn_history_rag_chunks.csv"
OUTPUT_PREVIEW_CSV = PROCESSED_DIR / "vn_history_rag_chunks_preview.csv"
OUTPUT_STATS = PROCESSED_DIR / "vn_history_rag_corpus_stats.json"
MANUAL_SFT_TEMPLATE_PATH = PROCESSED_DIR / "manual_sft_template.csv"

USE_HF_WIKIPEDIA = True
USE_WIKISOURCE = False

HF_DATASET_NAME = "DataStudio/Viet-wikipedia"
HF_SPLIT = "train"

# Đây là số record gốc quét từ HF, không phải số chunk.
MAX_HF_RECORDS = 250_000

CHUNK_WORDS = 350
CHUNK_OVERLAP = 60
MIN_CHUNK_WORDS = 80

# 4 = rộng hơn, nhiều noise hơn
# 6 = cân bằng, khuyên dùng
# 8 = sạch hơn, có thể bỏ sót
HISTORY_SCORE_THRESHOLD = 6

WIKISOURCE_URLS = [
    # "https://vi.wikisource.org/wiki/Vi%E1%BB%87t_Nam_s%E1%BB%AD_l%C6%B0%E1%BB%A3c/Quy%E1%BB%83n_I",
]

print("DRIVE_ROOT:", DRIVE_ROOT)
print("PROJECT_DIR:", PROJECT_DIR)
print("OUTPUT_JSONL:", OUTPUT_JSONL)

DRIVE_ROOT: /content/drive/MyDrive/vn_history_model_backups
PROJECT_DIR: /content/drive/MyDrive/vn_history_model_backups/rag_corpus_vn_history
OUTPUT_JSONL: /content/drive/MyDrive/vn_history_model_backups/rag_corpus_vn_history/processed/vn_history_rag_chunks.jsonl


## 3. Keyword filter + toàn bộ helper functions

In [4]:
# ============================================================
# Vietnamese History Keyword Filter — khoảng 905–2025
# Tất cả hàm quan trọng nằm trong cell này để tránh lỗi not defined.
# ============================================================

HISTORY_STRONG_KEYWORDS = [
    # 905–938
    "khúc thừa dụ", "khúc hạo", "khúc thừa mỹ", "họ khúc", "tiết độ sứ",
    "dương đình nghệ", "kiều công tiễn", "ngô quyền", "dương tam kha",
    "nam hán", "lưu hoằng tháo", "bạch đằng 938", "trận bạch đằng năm 938",

    # Ngô, Đinh, Tiền Lê
    "nhà ngô", "loạn 12 sứ quân", "mười hai sứ quân",
    "đinh bộ lĩnh", "đinh tiên hoàng", "đinh liễn", "đinh phế đế",
    "đại cồ việt", "hoa lư", "lê hoàn", "lê đại hành", "dương vân nga",
    "nhà tiền lê", "lê long đĩnh", "lê trung tông",
    "kháng chiến chống tống năm 981", "bạch đằng 981",

    # Lý
    "nhà lý", "lý công uẩn", "lý thái tổ", "chiếu dời đô", "thăng long",
    "lý thái tông", "lý thánh tông", "lý nhân tông", "lý thần tông",
    "lý anh tông", "lý cao tông", "lý huệ tông", "lý chiêu hoàng",
    "lý thường kiệt", "tông đản", "ỷ lan", "nguyên phi ỷ lan",
    "nam quốc sơn hà", "sông như nguyệt", "phòng tuyến sông như nguyệt",
    "kháng chiến chống tống 1075", "kháng chiến chống tống 1077",
    "văn miếu", "quốc tử giám", "vân đồn",

    # Trần, Mông - Nguyên
    "nhà trần", "trần thái tông", "trần thánh tông", "trần nhân tông",
    "trần anh tông", "trần minh tông", "trần nghệ tông", "trần duệ tông",
    "trần thủ độ", "trần thị dung", "trần hưng đạo", "trần quốc tuấn",
    "hưng đạo vương", "trần quang khải", "trần khánh dư", "trần nhật duật",
    "phạm ngũ lão", "trần bình trọng", "trần quốc toản",
    "hội nghị diên hồng", "hịch tướng sĩ", "bình than", "vạn kiếp",
    "kháng chiến chống mông nguyên", "quân mông nguyên", "quân nguyên",
    "quân mông", "trận đông bộ đầu", "trận hàm tử", "trận chương dương",
    "trận tây kết", "trận vạn kiếp", "bạch đằng 1288",
    "trận bạch đằng năm 1288", "thoát hoan", "toa đô", "ô mã nhi",
    "ngột lương hợp thai", "hốt tất liệt", "thiền phái trúc lâm",

    # Hồ, Minh thuộc
    "nhà hồ", "hồ quý ly", "hồ hán thương", "tây đô", "thành nhà hồ",
    "cải cách hồ quý ly", "đại ngu", "minh thuộc", "bắc thuộc lần thứ tư",
    "nhà minh", "quân minh", "trương phụ", "mộc thạnh", "hoàng phúc",
    "giản định đế", "trùng quang đế", "nhà hậu trần",

    # Lam Sơn, Lê sơ
    "khởi nghĩa lam sơn", "lam sơn", "lê lợi", "bình định vương",
    "lê thái tổ", "nguyễn trãi", "bình ngô đại cáo", "lê lai",
    "đinh lễ", "nguyễn xí", "trần nguyên hãn", "nguyễn chích",
    "trận tốt động chúc động", "tốt động chúc động",
    "trận chi lăng xương giang", "chi lăng", "xương giang",
    "vương thông", "liễu thăng", "lê sơ", "nhà hậu lê",
    "lê thái tông", "lê nhân tông", "lê thánh tông",
    "hồng đức", "luật hồng đức", "quốc triều hình luật",
    "hồng đức bản đồ", "đại việt sử ký toàn thư", "ngô sĩ liên",
    "lương thế vinh", "thân nhân trung",

    # Mạc, Lê trung hưng, Trịnh - Nguyễn
    "nhà mạc", "mạc đăng dung", "mạc đăng doanh", "mạc phúc hải",
    "mạc mậu hợp", "nam bắc triều", "lê trung hưng", "lê trang tông",
    "nguyễn kim", "trịnh kiểm", "trịnh tùng", "trịnh tráng",
    "chúa trịnh", "phủ chúa trịnh", "chúa nguyễn", "nguyễn hoàng",
    "chúa tiên", "nguyễn phúc nguyên", "nguyễn phúc tần",
    "nguyễn phúc chu", "nguyễn phúc khoát", "đàng ngoài", "đàng trong",
    "trịnh nguyễn phân tranh", "sông gianh", "lũy thầy", "đào duy từ",
    "phùng khắc khoan", "lê quý đôn", "phố hiến", "hội an",

    # Nam tiến, Champa, Chân Lạp
    "nam tiến", "champa", "chiêm thành", "chế bồng nga", "chế mân",
    "huyền trân công chúa", "thuận hóa", "quảng nam", "gia định",
    "chân lạp", "thủy chân lạp", "mạc cửu", "hà tiên", "nguyễn hữu cảnh",

    # Tây Sơn
    "tây sơn", "nhà tây sơn", "khởi nghĩa tây sơn",
    "nguyễn nhạc", "nguyễn huệ", "nguyễn lữ", "quang trung",
    "bắc bình vương", "ngọc hồi", "đống đa",
    "trận ngọc hồi đống đa", "trận rạch gầm xoài mút",
    "rạch gầm xoài mút", "quân xiêm", "quân thanh",
    "lê chiêu thống", "tôn sĩ nghị", "phú xuân",

    # Nguyễn, Pháp xâm lược
    "nhà nguyễn", "nguyễn ánh", "gia long", "minh mạng",
    "thiệu trị", "tự đức", "dục đức", "hiệp hòa", "kiến phúc",
    "hàm nghi", "đồng khánh", "thành thái", "duy tân",
    "khải định", "bảo đại", "kinh đô huế", "đại nam thực lục",
    "hoàng việt luật lệ", "luật gia long", "cải cách minh mạng",
    "lục tỉnh nam kỳ", "nam kỳ lục tỉnh", "pháp xâm lược việt nam",
    "thực dân pháp", "liên quân pháp tây ban nha", "đà nẵng 1858",
    "gia định 1859", "hòa ước nhâm tuất", "hòa ước giáp tuất",
    "hiệp ước harmand", "hiệp ước patenôtre", "hòa ước quý mùi",
    "kinh thành huế thất thủ", "tôn thất thuyết", "phong trào cần vương",
    "chiếu cần vương", "phan đình phùng", "cao thắng",
    "khởi nghĩa ba đình", "khởi nghĩa bãi sậy", "khởi nghĩa hương khê",
    "nguyễn thiện thuật", "đinh công tráng", "hoàng hoa thám",
    "khởi nghĩa yên thế", "đề thám",

    # Đầu thế kỷ XX
    "phan bội châu", "phan châu trinh", "huỳnh thúc kháng",
    "lương văn can", "nguyễn thái học", "phạm hồng thái",
    "đông du", "duy tân hội", "đông kinh nghĩa thục",
    "việt nam quang phục hội", "việt nam quốc dân đảng",
    "khởi nghĩa yên bái", "phong trào duy tân",
    "nguyễn tất thành", "nguyễn ái quốc", "hồ chí minh",
    "bản yêu sách của nhân dân an nam", "đường kách mệnh",
    "hội việt nam cách mạng thanh niên", "tân việt cách mạng đảng",
    "đông dương cộng sản đảng", "an nam cộng sản đảng",
    "đông dương cộng sản liên đoàn", "đảng cộng sản việt nam",
    "đảng cộng sản đông dương", "xô viết nghệ tĩnh",

    # 1930–1945
    "mặt trận việt minh", "việt minh", "cao trào kháng nhật cứu nước",
    "nhật đảo chính pháp", "nạn đói 1945", "tổng khởi nghĩa tháng tám",
    "cách mạng tháng tám", "19 tháng 8", "ngày 19 tháng 8",
    "tuyên ngôn độc lập", "2 tháng 9", "ngày 2 tháng 9",
    "quảng trường ba đình", "việt nam dân chủ cộng hòa",
    "chính phủ lâm thời", "quốc dân đại hội tân trào",
    "tân trào", "võ nguyên giáp", "trần phú", "lê hồng phong",
    "nguyễn văn cừ", "trường chinh", "phạm văn đồng",

    # 1945–1954
    "toàn quốc kháng chiến", "lời kêu gọi toàn quốc kháng chiến",
    "kháng chiến chống pháp", "chiến tranh đông dương",
    "hiệp định sơ bộ", "tạm ước 14 tháng 9",
    "chiến dịch việt bắc", "việt bắc thu đông 1947",
    "chiến dịch biên giới", "biên giới thu đông 1950",
    "đường số 4", "cao bằng", "đông khê",
    "chiến dịch hòa bình", "chiến dịch tây bắc", "chiến dịch thượng lào",
    "chiến dịch điện biên phủ", "điện biên phủ", "trận điện biên phủ",
    "de castries", "henri navarre", "kế hoạch navarre",
    "hiệp định geneva", "hiệp định giơnevơ", "geneva 1954",

    # 1954–1975
    "việt nam cộng hòa", "việt nam dân chủ cộng hòa",
    "chính quyền sài gòn", "ngô đình diệm", "ngô đình nhu",
    "dương văn minh", "nguyễn văn thiệu", "nguyễn cao kỳ",
    "mặt trận dân tộc giải phóng miền nam việt nam",
    "mặt trận giải phóng miền nam", "việt cộng",
    "chiến tranh việt nam", "kháng chiến chống mỹ",
    "đường trường sơn", "đường hồ chí minh", "đoàn 559",
    "ấp chiến lược", "đồng khởi", "phong trào đồng khởi",
    "bến tre 1960", "chiến tranh đặc biệt", "chiến tranh cục bộ",
    "việt nam hóa chiến tranh", "mậu thân 1968", "tổng tiến công mậu thân",
    "chiến dịch đường 9 nam lào", "lam sơn 719",
    "chiến dịch hồ chí minh", "tổng tiến công và nổi dậy mùa xuân 1975",
    "mùa xuân 1975", "chiến dịch tây nguyên", "buôn ma thuột",
    "chiến dịch huế đà nẵng", "chiến dịch xuân lộc",
    "30 tháng 4", "ngày 30 tháng 4", "dinh độc lập",
    "hiệp định paris", "paris 1973", "lê đức thọ", "henry kissinger",

    # 1975–1986
    "cộng hòa xã hội chủ nghĩa việt nam", "thống nhất đất nước",
    "quốc hội khóa vi", "sài gòn gia định", "thành phố hồ chí minh",
    "cải tạo công thương nghiệp", "kinh tế kế hoạch hóa",
    "chiến tranh biên giới tây nam", "khmer đỏ", "pol pot",
    "campuchia", "mặt trận đoàn kết dân tộc cứu nước campuchia",
    "chiến tranh biên giới việt trung", "chiến tranh biên giới phía bắc",
    "biên giới phía bắc 1979", "vị xuyên", "hà giang 1984",
    "gạc ma", "hải chiến trường sa 1988",

    # 1986–2025
    "đổi mới", "đại hội vi", "nguyễn văn linh", "võ văn kiệt",
    "kinh tế thị trường định hướng xã hội chủ nghĩa",
    "bình thường hóa quan hệ việt nam hoa kỳ",
    "việt nam gia nhập asean", "việt nam gia nhập wto",
    "hiệp định thương mại việt mỹ", "afta", "apec việt nam",
    "trương sa", "hoàng sa", "biển đông", "chủ quyền biển đảo",
    "giàn khoan hải dương 981", "hd-981",
    "covid-19 tại việt nam", "đại dịch covid-19 tại việt nam",
    "đại hội xiii", "nguyễn phú trọng", "nguyễn xuân phúc",
    "võ văn thưởng", "tô lâm", "phạm minh chính",
]

HISTORY_MEDIUM_KEYWORDS = [
    "lịch sử", "sử học", "sử liệu", "biên niên sử", "quốc sử",
    "triều đại", "vương triều", "hoàng đế", "vua", "chúa", "thái hậu",
    "tướng lĩnh", "danh tướng", "anh hùng dân tộc",
    "khởi nghĩa", "phong trào", "cách mạng", "chiến dịch", "trận đánh",
    "chiến thắng", "thất bại", "hòa ước", "hiệp định", "hiệp ước",
    "cải cách", "cải tổ", "bang giao", "ngoại giao", "triều cống",
    "xâm lược", "đô hộ", "thuộc địa", "bảo hộ", "giải phóng",
    "độc lập", "tự chủ", "thống nhất", "chia cắt",
    "đại việt", "đại cồ việt", "đại nam", "an nam", "giao chỉ", "giao châu",
    "bắc kỳ", "trung kỳ", "nam kỳ", "đàng ngoài", "đàng trong",
    "thăng long", "đông đô", "đông kinh", "hoa lư", "phú xuân", "huế",
    "sài gòn", "gia định", "hà nội", "hải phòng", "đà nẵng",
    "việt nam dân chủ cộng hòa", "việt nam cộng hòa",
    "cộng hòa xã hội chủ nghĩa việt nam", "chính phủ cách mạng lâm thời",
    "mặt trận tổ quốc việt nam", "quân đội nhân dân việt nam",
    "quân lực việt nam cộng hòa",
]

HISTORY_WEAK_KEYWORDS = [
    "việt nam", "người việt", "dân tộc việt", "đất nước", "nhà nước",
    "quân đội", "chính quyền", "chính phủ", "đảng", "quốc hội",
    "miền bắc", "miền nam", "miền trung",
]

BAD_TITLE_PATTERNS = [
    "danh sách", "thể loại:", "bản mẫu:", "tập tin:", "module:",
    "trợ giúp:", "wikipedia:", "định hướng", "category:", "template:",
    "file:", "portal:", "mediawiki:", "chủ đề:", "dự án:",
    "list of", "draft:", "user:", "talk:",
]

BAD_TITLE_EXACT_OR_CONTAINS = [
    "tiếng việt",
    "ngữ pháp tiếng việt",
    "địa lý việt nam",
    "khí hậu việt nam",
    "ẩm thực việt nam",
    "âm nhạc việt nam",
    "điện ảnh việt nam",
    "bóng đá việt nam",
]

def clean_text(text: str) -> str:
    if text is None:
        return ""
    text = str(text)
    text = text.replace("\xa0", " ")
    text = re.sub(r"\[[0-9]+\]", "", text)
    text = re.sub(r"\{\{.*?\}\}", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def get_field(record: Dict[str, Any], candidates: List[str], default: str = "") -> str:
    for c in candidates:
        if c in record and record[c] is not None:
            return str(record[c])
    return default

def stable_hash(text: str, n: int = 12) -> str:
    return hashlib.md5(text.encode("utf-8")).hexdigest()[:n]

def slugify_vn(text: str, max_len: int = 80) -> str:
    text = (text or "").lower().strip()
    text = re.sub(r"[^\w\s-]", "", text, flags=re.UNICODE)
    text = re.sub(r"\s+", "_", text)
    text = text[:max_len].strip("_")
    return text or "untitled"

def keyword_score(title: str, text: str) -> int:
    title_l = (title or "").lower()
    text_l = (text or "").lower()
    head = text_l[:6000]

    score = 0

    for k in HISTORY_STRONG_KEYWORDS:
        if k in title_l:
            score += 8
        elif k in head:
            score += 4

    for k in HISTORY_MEDIUM_KEYWORDS:
        if k in title_l:
            score += 4
        elif k in head:
            score += 2

    for k in HISTORY_WEAK_KEYWORDS:
        if k in title_l:
            score += 1
        elif k in head:
            score += 1

    years = re.findall(r"\b(9[0-9]{2}|1[0-9]{3}|20[0-2][0-9])\b", title_l + " " + head)
    if years:
        score += min(len(set(years)), 5)

    return score

def looks_like_history(title: str, text: str) -> bool:
    title_l = (title or "").lower()

    if any(p in title_l for p in BAD_TITLE_PATTERNS):
        return False

    if any(p in title_l for p in BAD_TITLE_EXACT_OR_CONTAINS):
        return False

    return keyword_score(title, text) >= HISTORY_SCORE_THRESHOLD

def word_chunks(text: str, chunk_words: int = 350, overlap: int = 60) -> List[str]:
    words = text.split()

    if len(words) < MIN_CHUNK_WORDS:
        return []

    if len(words) <= chunk_words:
        return [" ".join(words)]

    chunks = []
    start = 0

    while start < len(words):
        end = min(start + chunk_words, len(words))
        chunk = " ".join(words[start:end]).strip()

        if len(chunk.split()) >= MIN_CHUNK_WORDS:
            chunks.append(chunk)

        if end == len(words):
            break

        start = max(0, end - overlap)

    return chunks

def make_chunk_record(
    text: str,
    source: str,
    source_type: str,
    title: str,
    url: str = "",
    section: str = "",
    extra: Optional[Dict[str, Any]] = None,
    chunk_index: int = 0,
) -> Dict[str, Any]:
    base = f"{source}|{source_type}|{title}|{section}|{chunk_index}|{text[:120]}"
    chunk_id = f"{slugify_vn(source_type)}_{slugify_vn(title)}_{chunk_index:04d}_{stable_hash(base)}"

    rec = {
        "chunk_id": chunk_id,
        "source": source,
        "source_type": source_type,
        "title": title,
        "section": section,
        "url": url,
        "chunk_index": chunk_index,
        "text": text,
        "char_len": len(text),
        "word_len": len(text.split()),
    }

    if extra:
        rec.update(extra)

    return rec

def save_jsonl(records: List[Dict[str, Any]], path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)

    with path.open("w", encoding="utf-8") as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

    print(f"Saved {len(records):,} records to: {path}")

def save_stats(records: List[Dict[str, Any]], path: Path):
    if not records:
        stats = {"num_chunks": 0}
    else:
        df = pd.DataFrame(records)
        stats = {
            "num_chunks": int(len(df)),
            "num_sources": int(df["source"].nunique()) if "source" in df else None,
            "num_titles": int(df["title"].nunique()) if "title" in df else None,
            "source_type_counts": df["source_type"].value_counts().to_dict() if "source_type" in df else {},
            "word_len": {
                "min": int(df["word_len"].min()),
                "max": int(df["word_len"].max()),
                "mean": float(df["word_len"].mean()),
                "median": float(df["word_len"].median()),
            },
        }

    with path.open("w", encoding="utf-8") as f:
        json.dump(stats, f, ensure_ascii=False, indent=2)

    print(f"Saved stats to: {path}")
    return stats

def dedupe_chunks(chunks: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    seen = set()
    out = []

    for r in chunks:
        key = stable_hash(r["text"], n=16)

        if key in seen:
            continue

        seen.add(key)
        r["text_hash"] = key
        out.append(r)

    return out

print("Helper functions loaded OK.")

Helper functions loaded OK.


## 4. Preview cấu trúc Hugging Face dataset

In [5]:
if USE_HF_WIKIPEDIA:
    from datasets import load_dataset

    preview_ds = load_dataset(HF_DATASET_NAME, split=HF_SPLIT, streaming=True)

    preview = []
    for i, rec in enumerate(preview_ds):
        preview.append(rec)
        if i >= 2:
            break

    print("Dataset:", HF_DATASET_NAME)
    print("Preview records:", len(preview))

    for i, rec in enumerate(preview):
        print("\n" + "=" * 100)
        print("Record", i)
        print("Keys:", list(rec.keys()))

        for k, v in rec.items():
            s = str(v)
            print(f"- {k}: {s[:500]}")

README.md:   0%|          | 0.00/562 [00:00<?, ?B/s]

Dataset: DataStudio/Viet-wikipedia
Preview records: 3

Record 0
Keys: ['id', 'url', 'title', 'text']
- id: 4
- url: https://vi.wikipedia.org/wiki/Internet%20Society
- title: Internet Society
- text: Internet Society hay ISOC là một tổ chức quốc tế hoạt động phi lợi nhuận, phi chính phủ và bao gồm các thành viên có trình độ chuyên ngành. Tổ chức này chú trọng đến: tiêu chuẩn, giáo dục và các vấn đề về chính sách. Với trên 145 tổ chức thành viên và 65.000 thành viên cá nhân, ISOC bao gồm những con người cụ thể trong cộng đồng Internet. Mọi chi tiết có thể tìm thấy tại website của ISOC.

Internet Society nằm ở gần thủ đô Washington, DC, Hoa Kỳ và Geneva, Thụy Sĩ. Số hội viên của nó bao gồm hơn

Record 1
Keys: ['id', 'url', 'title', 'text']
- id: 13
- url: https://vi.wikipedia.org/wiki/Ti%E1%BA%BFng%20Vi%E1%BB%87t
- title: Tiếng Việt
- text: Tiếng Việt, cũng gọi là tiếng Việt Nam hay Việt ngữ là ngôn ngữ của người Việt và là ngôn ngữ chính thức tại Việt Nam. Đây là tiếng mẹ đẻ của khoảng 8

## 5. Build corpus từ Hugging Face Vietnamese Wikipedia

In [6]:
def build_from_hf_wikipedia(
    dataset_name: str,
    split: str,
    max_records: Optional[int] = None,
) -> List[Dict[str, Any]]:
    from datasets import load_dataset

    ds = load_dataset(dataset_name, split=split, streaming=True)

    chunks_out = []
    scanned = 0
    kept_docs = 0
    skipped_short = 0
    skipped_not_history = 0

    for rec in tqdm(ds, desc=f"Streaming {dataset_name}"):
        scanned += 1

        title = clean_text(get_field(rec, ["title", "doc_title", "page_title", "name"], ""))
        text = clean_text(get_field(rec, ["text", "content", "article", "document"], ""))
        url = get_field(rec, ["url", "source", "page_url"], "")

        if not text or len(text.split()) < MIN_CHUNK_WORDS:
            skipped_short += 1
            if max_records and scanned >= max_records:
                break
            continue

        if not looks_like_history(title, text):
            skipped_not_history += 1
            if max_records and scanned >= max_records:
                break
            continue

        kept_docs += 1
        score = keyword_score(title, text)
        doc_chunks = word_chunks(text, CHUNK_WORDS, CHUNK_OVERLAP)

        for i, ch in enumerate(doc_chunks):
            chunks_out.append(
                make_chunk_record(
                    text=ch,
                    source=dataset_name,
                    source_type="hf_wikipedia",
                    title=title or f"record_{scanned}",
                    url=url,
                    section="",
                    chunk_index=i,
                    extra={
                        "raw_record_index": scanned,
                        "hf_dataset": dataset_name,
                        "history_score": score,
                    },
                )
            )

        if max_records and scanned >= max_records:
            break

    print("\nHF build summary")
    print("Scanned:", scanned)
    print("Kept docs:", kept_docs)
    print("Skipped short:", skipped_short)
    print("Skipped not history:", skipped_not_history)
    print("Created chunks:", len(chunks_out))

    return chunks_out

all_chunks = []

if USE_HF_WIKIPEDIA:
    hf_chunks = build_from_hf_wikipedia(
        dataset_name=HF_DATASET_NAME,
        split=HF_SPLIT,
        max_records=MAX_HF_RECORDS,
    )
    all_chunks.extend(hf_chunks)

print("Total chunks so far:", len(all_chunks))

Streaming DataStudio/Viet-wikipedia: 0it [00:00, ?it/s]


HF build summary
Scanned: 250000
Kept docs: 37294
Skipped short: 142383
Skipped not history: 70323
Created chunks: 228646
Total chunks so far: 228646


## 6. Optional: Build corpus từ Wikisource

In [7]:
def fetch_wikisource_text(url: str) -> Dict[str, str]:
    import requests
    from bs4 import BeautifulSoup

    headers = {"User-Agent": "vn-history-rag-corpus-builder/0.1"}
    r = requests.get(url, headers=headers, timeout=40)
    r.raise_for_status()

    soup = BeautifulSoup(r.text, "html.parser")

    title_el = soup.select_one("h1#firstHeading")
    title = clean_text(title_el.get_text(" ")) if title_el else url

    content = soup.select_one("div.mw-parser-output")
    if not content:
        return {"title": title, "text": "", "url": url}

    for bad in content.select("table, .navbox, .metadata, .mw-editsection, style, script, sup.reference"):
        bad.decompose()

    parts = []
    for el in content.find_all(["h2", "h3", "h4", "p", "li"]):
        t = clean_text(el.get_text(" "))
        if not t or len(t) < 20:
            continue

        if el.name in ["h2", "h3", "h4"]:
            parts.append(f"\n## {t}\n")
        else:
            parts.append(t)

    text = "\n".join(parts)
    return {"title": title, "text": text, "url": url}

def build_from_wikisource_urls(urls: List[str]) -> List[Dict[str, Any]]:
    chunks_out = []

    for url in tqdm(urls, desc="Crawling Wikisource"):
        try:
            doc = fetch_wikisource_text(url)
        except Exception as e:
            print("ERROR:", url, e)
            continue

        text = clean_text(doc["text"])

        if not text or len(text.split()) < MIN_CHUNK_WORDS:
            print("Empty/short:", url)
            continue

        score = keyword_score(doc["title"], text)
        doc_chunks = word_chunks(text, CHUNK_WORDS, CHUNK_OVERLAP)

        for i, ch in enumerate(doc_chunks):
            chunks_out.append(
                make_chunk_record(
                    text=ch,
                    source=doc["title"],
                    source_type="wikisource",
                    title=doc["title"],
                    url=doc["url"],
                    section="",
                    chunk_index=i,
                    extra={
                        "original_url": url,
                        "history_score": score,
                    },
                )
            )

        time.sleep(0.5)

    print("Wikisource chunks:", len(chunks_out))
    return chunks_out

if USE_WIKISOURCE:
    wiki_chunks = build_from_wikisource_urls(WIKISOURCE_URLS)
    all_chunks.extend(wiki_chunks)

print("Total chunks:", len(all_chunks))

Total chunks: 228646


## 7. Deduplicate và lưu JSONL/CSV

In [8]:
before = len(all_chunks)
all_chunks = dedupe_chunks(all_chunks)
after = len(all_chunks)

print(f"Deduped chunks: {before:,} -> {after:,}")

save_jsonl(all_chunks, OUTPUT_JSONL)

if all_chunks:
    df = pd.DataFrame(all_chunks)

    # CSV đầy đủ
    df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
    print(f"Saved full CSV to: {OUTPUT_CSV}")

    # CSV preview
    df_preview = df.copy()
    df_preview["text_preview"] = df_preview["text"].astype(str).str.slice(0, 700)

    preview_cols = [
        "chunk_id", "source_type", "source", "title", "url",
        "chunk_index", "word_len", "char_len", "history_score", "text_preview"
    ]
    preview_cols = [c for c in preview_cols if c in df_preview.columns]

    df_preview[preview_cols].to_csv(OUTPUT_PREVIEW_CSV, index=False, encoding="utf-8-sig")
    print(f"Saved preview CSV to: {OUTPUT_PREVIEW_CSV}")

    stats = save_stats(all_chunks, OUTPUT_STATS)
    print(json.dumps(stats, ensure_ascii=False, indent=2))
else:
    print("No chunks to save.")

Deduped chunks: 228,646 -> 228,646
Saved 228,646 records to: /content/drive/MyDrive/vn_history_model_backups/rag_corpus_vn_history/processed/vn_history_rag_chunks.jsonl
Saved full CSV to: /content/drive/MyDrive/vn_history_model_backups/rag_corpus_vn_history/processed/vn_history_rag_chunks.csv
Saved preview CSV to: /content/drive/MyDrive/vn_history_model_backups/rag_corpus_vn_history/processed/vn_history_rag_chunks_preview.csv
Saved stats to: /content/drive/MyDrive/vn_history_model_backups/rag_corpus_vn_history/processed/vn_history_rag_corpus_stats.json
{
  "num_chunks": 228646,
  "num_sources": 1,
  "num_titles": 37294,
  "source_type_counts": {
    "hf_wikipedia": 228646
  },
  "word_len": {
    "min": 80,
    "max": 350,
    "mean": 328.88501001548246,
    "median": 350.0
  }
}


## 8. Validate JSONL sạch

In [9]:
bad_lines = []
num_lines = 0

with open(OUTPUT_JSONL, "r", encoding="utf-8") as f:
    for line_no, line in enumerate(f, start=1):
        num_lines += 1
        try:
            obj = json.loads(line)
            if not isinstance(obj, dict):
                bad_lines.append((line_no, "Line is not a JSON object", line[:200]))
        except Exception as e:
            bad_lines.append((line_no, str(e), line[:200]))

print("JSONL path:", OUTPUT_JSONL)
print("Total lines:", num_lines)
print("Bad lines:", len(bad_lines))

if bad_lines:
    print("First bad lines:")
    for item in bad_lines[:5]:
        print(item)
else:
    print("OK: JSONL từng dòng hợp lệ.")

df_test = pd.read_json(OUTPUT_JSONL, lines=True)
print("Pandas read OK. Shape:", df_test.shape)
display(df_test.head(5))

JSONL path: /content/drive/MyDrive/vn_history_model_backups/rag_corpus_vn_history/processed/vn_history_rag_chunks.jsonl
Total lines: 228646
Bad lines: 0
OK: JSONL từng dòng hợp lệ.
Pandas read OK. Shape: (228646, 14)


,chunk_id,source,source_type,title,section,url,chunk_index,text,char_len,word_len,raw_record_index,hf_dataset,history_score,text_hash
0,hf_wikipedia_internet_society_0000_f1c970121556,DataStudio/Viet-wikipedia,hf_wikipedia,Internet Society,,https://vi.wikipedia.org/wiki/Internet%20Society,0,Internet Society hay ISOC là một tổ chức quốc ...,1204,243,1,DataStudio/Viet-wikipedia,6,3ba0424fff55eeaa
1,hf_wikipedia_ohio_0000_c99496304b2d,DataStudio/Viet-wikipedia,hf_wikipedia,Ohio,,https://vi.wikipedia.org/wiki/Ohio,0,"Ohio (viết tắt là OH, viết tắt cũ là O.) là mộ...",1715,350,3,DataStudio/Viet-wikipedia,20,021eeb0ff6b98084
2,hf_wikipedia_ohio_0001_27762088ec7a,DataStudio/Viet-wikipedia,hf_wikipedia,Ohio,,https://vi.wikipedia.org/wiki/Ohio,1,1787. Ohio nằm trong vùng lãnh thổ Tây Bắc. Vù...,1688,350,3,DataStudio/Viet-wikipedia,20,b0eb5a9d92fa3265
3,hf_wikipedia_ohio_0002_a0ec6c2b58ad,DataStudio/Viet-wikipedia,hf_wikipedia,Ohio,,https://vi.wikipedia.org/wiki/Ohio,2,Ontario của Canada). Ohio tiếp giáp với Pennsy...,1664,350,3,DataStudio/Viet-wikipedia,20,0ac652d985e9792d
4,hf_wikipedia_ohio_0003_e5cc03d40f67,DataStudio/Viet-wikipedia,hf_wikipedia,Ohio,,https://vi.wikipedia.org/wiki/Ohio,3,thu hút được sự quan tâm đặc biệt về lịch sử. ...,1699,350,3,DataStudio/Viet-wikipedia,20,a9f036dc956d5dbd


## 9. Xem nhanh output

In [10]:
if all_chunks:
    df = pd.DataFrame(all_chunks)
    print("Shape:", df.shape)

    df_view = df.copy()
    df_view["text_preview"] = df_view["text"].astype(str).str.slice(0, 700)

    cols_show = [
        "chunk_id", "source_type", "source", "title", "url",
        "chunk_index", "word_len", "char_len", "history_score", "text_preview"
    ]
    cols_show = [c for c in cols_show if c in df_view.columns]

    display(df_view[cols_show].head(20))

    print("\nWord length stats:")
    display(df["word_len"].describe())

    print("\nSource type counts:")
    display(df["source_type"].value_counts())

    if "history_score" in df.columns:
        print("\nHistory score stats:")
        display(df["history_score"].describe())
else:
    print("No chunks.")

Shape: (228646, 14)


,chunk_id,source_type,source,title,url,chunk_index,word_len,char_len,history_score,text_preview
0,hf_wikipedia_internet_society_0000_f1c970121556,hf_wikipedia,DataStudio/Viet-wikipedia,Internet Society,https://vi.wikipedia.org/wiki/Internet%20Society,0,243,1204,6,Internet Society hay ISOC là một tổ chức quốc ...
1,hf_wikipedia_ohio_0000_c99496304b2d,hf_wikipedia,DataStudio/Viet-wikipedia,Ohio,https://vi.wikipedia.org/wiki/Ohio,0,350,1715,20,"Ohio (viết tắt là OH, viết tắt cũ là O.) là mộ..."
2,hf_wikipedia_ohio_0001_27762088ec7a,hf_wikipedia,DataStudio/Viet-wikipedia,Ohio,https://vi.wikipedia.org/wiki/Ohio,1,350,1688,20,1787. Ohio nằm trong vùng lãnh thổ Tây Bắc. Vù...
3,hf_wikipedia_ohio_0002_a0ec6c2b58ad,hf_wikipedia,DataStudio/Viet-wikipedia,Ohio,https://vi.wikipedia.org/wiki/Ohio,2,350,1664,20,Ontario của Canada). Ohio tiếp giáp với Pennsy...
4,hf_wikipedia_ohio_0003_e5cc03d40f67,hf_wikipedia,DataStudio/Viet-wikipedia,Ohio,https://vi.wikipedia.org/wiki/Ohio,3,350,1699,20,thu hút được sự quan tâm đặc biệt về lịch sử. ...
5,hf_wikipedia_ohio_0004_9798bed24e85,hf_wikipedia,DataStudio/Viet-wikipedia,Ohio,https://vi.wikipedia.org/wiki/Ohio,4,114,525,20,đại học công lập và khu vực. 46 trường nghệ th...
6,hf_wikipedia_california_0000_b2cb2058f797,hf_wikipedia,DataStudio/Viet-wikipedia,California,https://vi.wikipedia.org/wiki/California,0,350,1619,13,California (còn được người Việt gọi vắn tắt là...
7,hf_wikipedia_california_0001_27ba091271d2,hf_wikipedia,DataStudio/Viet-wikipedia,California,https://vi.wikipedia.org/wiki/California,1,350,1598,13,"và Khu vực Vịnh San Francisco (9,6 triệu người..."
8,hf_wikipedia_california_0002_2e844906660c,hf_wikipedia,DataStudio/Viet-wikipedia,California,https://vi.wikipedia.org/wiki/California,2,350,1679,13,"nhân vật trong các lĩnh vực truyền thông, công..."
9,hf_wikipedia_california_0003_feefb198e5a1,hf_wikipedia,DataStudio/Viet-wikipedia,California,https://vi.wikipedia.org/wiki/California,3,350,1681,13,"California kề cận với Thái Bình Dương, Oregon,..."



Word length stats:


,word_len
count,228646.000000
mean,328.885010
std,58.081031
min,80.000000
25%,350.000000
50%,350.000000
75%,350.000000
max,350.000000



Source type counts:


,count
source_type,
hf_wikipedia,228646



History score stats:


,history_score
count,228646.000000
mean,22.010912
std,21.291854
min,6.000000
25%,9.000000
50%,14.000000
75%,25.000000
max,273.000000


## 10. Search nhanh trong corpus

In [11]:
keyword = "Bạch Đằng"

if all_chunks:
    df = pd.DataFrame(all_chunks)
    mask = (
        df["text"].astype(str).str.contains(keyword, case=False, na=False)
        | df["title"].astype(str).str.contains(keyword, case=False, na=False)
    )

    print("Keyword:", keyword)
    print("Số chunk tìm thấy:", int(mask.sum()))

    result = df.loc[mask].copy()

    if len(result):
        result["text_preview"] = result["text"].astype(str).str.slice(0, 900)
        cols_show = [
            "chunk_id", "source_type", "title", "url",
            "chunk_index", "word_len", "history_score", "text_preview"
        ]
        cols_show = [c for c in cols_show if c in result.columns]
        display(result[cols_show].head(20))
else:
    print("No chunks.")

Keyword: Bạch Đằng
Số chunk tìm thấy: 533


,chunk_id,source_type,title,url,chunk_index,word_len,history_score,text_preview
132,hf_wikipedia_thành_phố_hồ_chí_minh_0050_c18241...,hf_wikipedia,Thành phố Hồ Chí Minh,https://vi.wikipedia.org/wiki/Th%C3%A0nh%20ph%...,50,350,86,"tổng vốn đầu tư cao nhất, đội vốn nhiều nhất l..."
413,hf_wikipedia_hà_nội_0028_6ad50afce102,hf_wikipedia,Hà Nội,https://vi.wikipedia.org/wiki/H%C3%A0%20N%E1%B...,28,350,52,được một quần thể di tích đa dạng là Văn Miếu-...
414,hf_wikipedia_hà_nội_0029_9c2d6c5eda6a,hf_wikipedia,Hà Nội,https://vi.wikipedia.org/wiki/H%C3%A0%20N%E1%B...,29,350,52,"khu: nhượng địa, thành cũ và nam hồ Hoàn Kiếm,..."
656,hf_wikipedia_lý_thường_kiệt_0010_5e86ff6c2e85,hf_wikipedia,Lý Thường Kiệt,https://vi.wikipedia.org/wiki/L%C3%BD%20Th%C6%...,10,350,72,"ải Hà Nội, Hoàng Kim Mãn và Sầm Khánh Tân giữ ..."
657,hf_wikipedia_lý_thường_kiệt_0011_e24044aff242,hf_wikipedia,Lý Thường Kiệt,https://vi.wikipedia.org/wiki/L%C3%BD%20Th%C6%...,11,350,72,"Long, thì sông Bạch Đằng không can hệ, vì đã c..."
950,hf_wikipedia_trần_hưng_đạo_0001_189d18eabadb,hf_wikipedia,Trần Hưng Đạo,https://vi.wikipedia.org/wiki/Tr%E1%BA%A7n%20H...,1,350,108,"quân đội cả nước. Trên cương vị này, năm 1285,..."
962,hf_wikipedia_trần_hưng_đạo_0013_ad4738542073,hf_wikipedia,Trần Hưng Đạo,https://vi.wikipedia.org/wiki/Tr%E1%BA%A7n%20H...,13,350,108,đô mà tổ chức phòng thủ ngay tại Thăng Long. T...
970,hf_wikipedia_trần_hưng_đạo_0021_8b51a1da020e,hf_wikipedia,Trần Hưng Đạo,https://vi.wikipedia.org/wiki/Tr%E1%BA%A7n%20H...,21,350,108,"lời: ""Bệ hạ chém đầu tôi trước rồi hãy hàng gi..."
972,hf_wikipedia_trần_hưng_đạo_0023_3f191c36c2fe,hf_wikipedia,Trần Hưng Đạo,https://vi.wikipedia.org/wiki/Tr%E1%BA%A7n%20H...,23,350,108,"hậu của Trần Nhân Tông, mẹ đẻ của Trần Anh Tôn..."
974,hf_wikipedia_trần_hưng_đạo_0025_26fb79230516,hf_wikipedia,Trần Hưng Đạo,https://vi.wikipedia.org/wiki/Tr%E1%BA%A7n%20H...,25,298,108,"Đền thờ Đức Thánh Trần, thôn Lương Xá, xã Liên..."


## 11. Tạo manual SFT template

In [12]:
if all_chunks:
    rows = []
    for i, r in enumerate(all_chunks[:300]):
        rows.append({
            "sample_id": f"sample_{i+1:05d}",
            "type": "grounded_qa",
            "question": "",
            "context_ids": r["chunk_id"],
            "context_text": r["text"],
            "gold_answer": "",
            "gold_evidence": r["chunk_id"],
            "note": "",
        })

    template_df = pd.DataFrame(rows)
    template_df.to_csv(MANUAL_SFT_TEMPLATE_PATH, index=False, encoding="utf-8-sig")
    print(f"Saved manual SFT template to: {MANUAL_SFT_TEMPLATE_PATH}")
    display(template_df.head(10))
else:
    print("No chunks available.")

Saved manual SFT template to: /content/drive/MyDrive/vn_history_model_backups/rag_corpus_vn_history/processed/manual_sft_template.csv


,sample_id,type,question,context_ids,context_text,gold_answer,gold_evidence,note
0,sample_00001,grounded_qa,,hf_wikipedia_internet_society_0000_f1c970121556,Internet Society hay ISOC là một tổ chức quốc ...,,hf_wikipedia_internet_society_0000_f1c970121556,
1,sample_00002,grounded_qa,,hf_wikipedia_ohio_0000_c99496304b2d,"Ohio (viết tắt là OH, viết tắt cũ là O.) là mộ...",,hf_wikipedia_ohio_0000_c99496304b2d,
2,sample_00003,grounded_qa,,hf_wikipedia_ohio_0001_27762088ec7a,1787. Ohio nằm trong vùng lãnh thổ Tây Bắc. Vù...,,hf_wikipedia_ohio_0001_27762088ec7a,
3,sample_00004,grounded_qa,,hf_wikipedia_ohio_0002_a0ec6c2b58ad,Ontario của Canada). Ohio tiếp giáp với Pennsy...,,hf_wikipedia_ohio_0002_a0ec6c2b58ad,
4,sample_00005,grounded_qa,,hf_wikipedia_ohio_0003_e5cc03d40f67,thu hút được sự quan tâm đặc biệt về lịch sử. ...,,hf_wikipedia_ohio_0003_e5cc03d40f67,
5,sample_00006,grounded_qa,,hf_wikipedia_ohio_0004_9798bed24e85,đại học công lập và khu vực. 46 trường nghệ th...,,hf_wikipedia_ohio_0004_9798bed24e85,
6,sample_00007,grounded_qa,,hf_wikipedia_california_0000_b2cb2058f797,California (còn được người Việt gọi vắn tắt là...,,hf_wikipedia_california_0000_b2cb2058f797,
7,sample_00008,grounded_qa,,hf_wikipedia_california_0001_27ba091271d2,"và Khu vực Vịnh San Francisco (9,6 triệu người...",,hf_wikipedia_california_0001_27ba091271d2,
8,sample_00009,grounded_qa,,hf_wikipedia_california_0002_2e844906660c,"nhân vật trong các lĩnh vực truyền thông, công...",,hf_wikipedia_california_0002_2e844906660c,
9,sample_00010,grounded_qa,,hf_wikipedia_california_0003_feefb198e5a1,"California kề cận với Thái Bình Dương, Oregon,...",,hf_wikipedia_california_0003_feefb198e5a1,


## 12. Liệt kê file output

In [13]:
print("Processed files:")
for p in sorted(PROCESSED_DIR.glob("*")):
    print("-", p, "| size:", p.stat().st_size, "bytes")

Processed files:
- /content/drive/MyDrive/vn_history_model_backups/rag_corpus_vn_history/processed/manual_sft_template.csv | size: 656057 bytes
- /content/drive/MyDrive/vn_history_model_backups/rag_corpus_vn_history/processed/vn_history_rag_chunks.csv | size: 511019662 bytes
- /content/drive/MyDrive/vn_history_model_backups/rag_corpus_vn_history/processed/vn_history_rag_chunks.jsonl | size: 558115563 bytes
- /content/drive/MyDrive/vn_history_model_backups/rag_corpus_vn_history/processed/vn_history_rag_chunks_preview.csv | size: 249728958 bytes
- /content/drive/MyDrive/vn_history_model_backups/rag_corpus_vn_history/processed/vn_history_rag_corpus_stats.json | size: 231 bytes
